# Calyx of Held — Colab Edition

This notebook is a Google-Colab-friendly, Python/Jupyter port of a NEURON model of synaptic
transmission at the calyx of Held, a giant glutamatergic synapse in the mammalian auditory
brainstem, by Bruce Graham, Adrian Wong and Ian Forsythe. Original files:
https://modeldb.science/19747

> Forsythe ID, Graham BP, Wong AYC. A computational model of synaptic transmission at the
> calyx of Held. *Neurocomputing* 38-40:37-42, 2001.

The presynaptic terminal (`AXON`) contains a stochastic (Monte Carlo) model of vesicle
mobilization and release at 500 independent release sites (`COH` mechanism). Each site drives
its own postsynaptic AMPA receptor (`AMPA` mechanism, a 6-state kinetic scheme) on the
postsynaptic cell (`MNTB`). Two modes reuse this same synapse:

- **EPSC mode** (`run_epsc.hoc`): `MNTB` is voltage-clamped, so the recorded current directly
  shows facilitation and depression of the postsynaptic response during a spike train
  (paper figs. 2-3).
- **EPSP mode** (`run_epsp.hoc`): `MNTB` has active Na/K conductances (`BKDNaDR` mechanism)
  instead, showing the resulting postsynaptic voltage/spiking.

No local installation is needed — just run the cells top to bottom in Colab.

## 1. Setup (run once per Colab session)

Installs the `neuron` Python package, clones the original `.mod` mechanism files from the
[ModelDB GitHub mirror](https://github.com/ModelDBRepository/19747), and compiles them with
`nrnivmodl`.

In [ ]:
%%capture
!pip install neuron

In [ ]:
import os

MOD_SRC_DIR = "coh_src"

if not os.path.isdir(MOD_SRC_DIR):
    !git clone --depth 1 https://github.com/ModelDBRepository/19747.git {MOD_SRC_DIR}

!cd {MOD_SRC_DIR} && nrnivmodl

In [ ]:
from neuron import h
from neuron import load_mechanisms
import matplotlib.pyplot as plt

load_mechanisms(MOD_SRC_DIR)
h.load_file("stdrun.hoc")

print("NEURON is ready, mechanisms loaded from:", MOD_SRC_DIR)

## 2. Build the cell

A presynaptic terminal (`AXON`) and postsynaptic cell body (`MNTB`), both single compartments
(L=20 &mu;m, diam=20 &mu;m, Ra=200 &Omega;&middot;cm, passive leak). The calyx of Held (`COH`
point process) sits on `AXON` and drives `N=500` independent `AMPA` receptors on `MNTB`, one per
release site, connected via NEURON `setpointer` (mirrors `coh.T[i]` &rarr; `c[i].C` in the
original `prepost.hoc`). `BKDNaDR` (Na/K/leak) is inserted on `MNTB` for EPSP mode but starts
with zero conductance — it's only enabled by `run_epsp_experiment()`. Likewise `vcl` (`VClamp`)
is only activated by `run_epsc_experiment()`.

In [ ]:
# presynaptic terminal and postsynaptic cell body (single compartments, matches prepost.hoc)
AXON = h.Section(name="AXON")
MNTB = h.Section(name="MNTB")

for sec in (AXON, MNTB):
    sec.L = 20
    sec.diam = 20
    sec.Ra = 200
    sec.insert("pas")
    sec(0.5).g_pas = 1 / 50000

MNTB.insert("BKDNaDR")
MNTB(0.5).gnabar_BKDNaDR = 0.0
MNTB(0.5).gkbar_BKDNaDR = 0.0  # enabled only in EPSP mode

coh = h.COH(AXON(0.5))  # Monte Carlo model of vesicle mobilization/release

N = 500  # number of release sites, must match RSIZE in coh.mod
ampa_list = []
for i in range(N):
    a = h.AMPA(MNTB(0.5))
    a.gmax = 500  # pS
    h.setpointer(coh._ref_T[i], "C", a)
    ampa_list.append(a)

vcl = h.VClamp(MNTB(0.5))  # active only in EPSC mode (dur[0]=0 keeps it inactive otherwise)

print("AXON + MNTB built,", N, "AMPA receptors connected to the calyx of Held release sites")

## 3. Common recording

`t_vec`/`v_vec` track time and postsynaptic voltage; `i_vec` records the voltage-clamp current
(used in EPSC mode); `ntot_vec` records the total number of readily-releasable vesicles across
all release sites (a useful window into the facilitation/depression mechanism).

In [ ]:
t_vec = h.Vector().record(h._ref_t)
v_vec = h.Vector().record(MNTB(0.5)._ref_v)
i_vec = h.Vector().record(vcl._ref_i)
ntot_vec = h.Vector().record(coh._ref_ntot)

## 4. Shared simulation helpers

`DEFAULT_COH_PARAMS` mirrors the `COH (Globals)` panel from the original session files (release
probability, vesicle pool/replenishment rates, local/distant calcium transients, transmitter
pulse); `DEFAULT_SPIKE_PARAMS` mirrors the `Presynaptic Spikes` panel (stimulus + recovery-test
spike trains), i.e. the `prespikes()` procedure. Both EPSC and EPSP modes share these plus
`_plot_panels()`.

In [ ]:
# COH (Globals) panel defaults, from coh.mod / cohepsc.ses
DEFAULT_COH_PARAMS = dict(
    rseed=0, pv0=0.5, n0=1, ke=8, kd=0.0002,
    Camp=0.1, Cres=0, Cdur=1,
    Cnamp=0.01, Cnres=0, Cndur=2,
    Tamp=1, Tdur=1,
)

# Presynaptic Spikes panel defaults, from prespikes.hoc
DEFAULT_SPIKE_PARAMS = dict(
    stimdel=10, stimdur=100, stimfre=100,
    testdel=10, testdur=1000, testfre=20,
)

SSIZE = 500  # max presynaptic spikes (bounds check, matches prespikes.hoc)

def _apply_coh_params(p):
    # these PARAMETERs are not RANGE in coh.mod, so they're GLOBAL (set via h.xxx_COH, not coh.xxx)
    h.rseed_COH = p["rseed"]
    h.pv0_COH = p["pv0"]
    h.n0_COH = p["n0"]
    h.ke_COH = p["ke"]
    h.kd_COH = p["kd"]
    h.Camp_COH = p["Camp"]
    h.Cres_COH = p["Cres"]
    h.Cdur_COH = p["Cdur"]
    h.Cnamp_COH = p["Cnamp"]
    h.Cnres_COH = p["Cnres"]
    h.Cndur_COH = p["Cndur"]
    h.Tamp_COH = p["Tamp"]
    h.Tdur_COH = p["Tdur"]

def _set_spike_train(p):
    """Port of prespikes.hoc: a stimulus train followed by a recovery-test train."""
    sisi = 1000 / p["stimfre"]
    sn = int(p["stimdur"] / sisi)

    tn = 0
    if p["testfre"] > 0:
        tisi = 1000 / p["testfre"]
        tn = int(p["testdur"] / tisi)

    if sn + tn + 2 > SSIZE:
        print("Too many spikes requested!")
        return

    for j in range(sn + 1):
        coh.spike[j] = p["stimdel"] + j * sisi
    coh.spike[sn + 1] = 1e20

    if tn > 0:
        for j in range(tn + 1):
            coh.spike[j + sn + 1] = coh.spike[sn] + p["testdel"] + j * tisi
        coh.spike[tn + sn + 2] = 1e20

def _panel_data(spec):
    return spec["compute"]() if "compute" in spec else spec["vector"]

def _plot_panels(plot_panels, keys=None):
    keys = list(plot_panels.keys()) if keys is None else list(keys)
    if not keys:
        print("No panels selected.")
        return

    fig, axes = plt.subplots(len(keys), 1, figsize=(7, 2.5 * len(keys)), sharex=True)
    axes = [axes] if len(keys) == 1 else axes
    for ax, key in zip(axes, keys):
        spec = plot_panels[key]
        ax.plot(t_vec, _panel_data(spec))
        ax.set_ylabel(spec["ylabel"])
        if spec.get("ylim"):
            ax.set_ylim(*spec["ylim"])
    axes[-1].set_xlabel("time (ms)")
    plt.show()

## 5. Shared dashboard helpers

Same slider/reset/changed-checkbox and panel-visibility helpers used by both dashboards below.

In [ ]:
from ipywidgets import FloatSlider, Checkbox, Button, Dropdown, HBox, VBox, Output, Layout, Label
from IPython.display import display

def _display_value(param_ui, name, raw_value):
    return raw_value / param_ui[name].get("scale", 1)

def _raw_value(param_ui, name, display_value):
    return display_value * param_ui[name].get("scale", 1)

def _raw_values(param_ui, sliders):
    return {name: _raw_value(param_ui, name, slider.value) for name, slider in sliders.items()}

def _build_param_rows(param_ui, get_default):
    """get_default(name) -> current default/raw value for that parameter (may change over time)."""
    sliders, changed_flags, rows = {}, {}, []

    def _make_change_handler(name):
        def _on_change(change):
            changed_flags[name].value = (change["new"] != _display_value(param_ui, name, get_default(name)))
        return _on_change

    def _make_reset_handler(name):
        def _on_click(_btn):
            sliders[name].value = _display_value(param_ui, name, get_default(name))
        return _on_click

    for name, spec in param_ui.items():
        slider_kwargs = {k: v for k, v in spec.items() if k not in ("scale", "unit", "description")}
        slider = FloatSlider(value=_display_value(param_ui, name, get_default(name)), description=spec["description"],
                              layout=Layout(width="350px"), **slider_kwargs)
        unit_label = Label(value=spec.get("unit", ""), layout=Layout(width="110px"))
        changed = Checkbox(value=False, description="changed", disabled=True, indent=False,
                            layout=Layout(width="90px"))
        reset_btn = Button(description="reset", layout=Layout(width="60px"))

        sliders[name] = slider
        changed_flags[name] = changed
        slider.observe(_make_change_handler(name), names="value")
        reset_btn.on_click(_make_reset_handler(name))

        rows.append(HBox([slider, unit_label, changed, reset_btn]))

    return sliders, changed_flags, rows

def _build_panel_checks(plot_panels):
    checks = {key: Checkbox(value=True, description=spec["label"], indent=False, layout=Layout(width="90px"))
              for key, spec in plot_panels.items()}
    return checks, HBox([Label("Show:")] + list(checks.values()))

def _selected_panels(panel_checks):
    return [key for key, cb in panel_checks.items() if cb.value]

# 1. EPSC mode (voltage clamp — facilitation & depression)

`MNTB` is voltage-clamped at `v_init` throughout, so the recorded current directly reflects the
summed postsynaptic AMPA conductances driven by the calyx's stochastic vesicle release —
facilitation and depression appear as changes in EPSC amplitude across the stimulus train, and
the `test` spikes (after a delay) reveal how much the synapse has recovered. Sourced from
`run_epsc.hoc` / `cohepsc.ses`.

In [ ]:
DEFAULT_EPSC_PARAMS = {**DEFAULT_COH_PARAMS, **DEFAULT_SPIKE_PARAMS,
    "celsius": 36, "v_init": 0, "tstop": 100}

PLOT_PANELS_EPSC = {
    "i": dict(label="EPSC", vector=i_vec, ylabel="postsynaptic current (nA)"),
    "ntot": dict(label="vesicle pool", vector=ntot_vec, ylabel="available vesicles (ntot)"),
}

def run_epsc_experiment(panels=None, **overrides):
    p = {**DEFAULT_EPSC_PARAMS, **overrides}

    MNTB(0.5).g_pas = 1 / 50000
    MNTB(0.5).e_pas = p["v_init"]
    MNTB(0.5).gnabar_BKDNaDR = 0.0
    MNTB(0.5).gkbar_BKDNaDR = 0.0  # EPSP-mode spiking conductances must stay off here

    vcl.dur[0] = 11000  # clamp for effectively the whole run
    vcl.amp[0] = p["v_init"]

    _apply_coh_params(p)
    _set_spike_train(p)

    h.celsius = p["celsius"]
    h.tstop = p["tstop"]
    h.v_init = p["v_init"]
    h.run()

    _plot_panels(PLOT_PANELS_EPSC, panels)

In [ ]:
# @title Specifying default parameters

EPSC_PARAM_UI = {
    "stimdel": dict(min=0, max=50, step=1, description="stim delay", unit="ms"),
    "stimdur": dict(min=0, max=500, step=10, description="stim duration", unit="ms"),
    "stimfre": dict(min=1, max=300, step=1, description="stim frequency", unit="Hz"),
    "testdel": dict(min=0, max=50, step=1, description="test delay", unit="ms"),
    "testdur": dict(min=0, max=2000, step=10, description="test duration", unit="ms"),
    "testfre": dict(min=0, max=100, step=1, description="test frequency", unit="Hz"),
    "pv0": dict(min=0, max=1, step=0.01, description="release probability scale", unit=""),
    "n0": dict(min=0, max=5, step=1, description="initial RRVP size", unit="vesicles"),
    "ke": dict(min=0, max=20, step=0.5, description="enhanced replenish rate", unit="/mM/ms"),
    "kd": dict(min=0, max=0.01, step=0.0001, readout_format=".4f", description="background depletion", unit="/ms"),
    "Camp": dict(min=0, max=1, step=0.01, description="Ca transient amp (local)", unit="mM"),
    "Cres": dict(min=0, max=1, step=0.01, description="Ca residual (local)", unit="mM"),
    "Cdur": dict(min=0, max=10, step=0.1, description="Ca transient dur (local)", unit="ms"),
    "Cnamp": dict(min=0, max=0.1, step=0.001, readout_format=".3f", description="Ca transient amp (distant)", unit="mM"),
    "Cnres": dict(min=0, max=0.1, step=0.001, readout_format=".3f", description="Ca residual (distant)", unit="mM"),
    "Cndur": dict(min=0, max=20, step=0.5, description="Ca transient dur (distant)", unit="ms"),
    "Tamp": dict(min=0, max=5, step=0.1, description="transmitter pulse amp", unit="mM"),
    "Tdur": dict(min=0, max=10, step=0.1, description="transmitter pulse dur", unit="ms"),
    "rseed": dict(min=0, max=100, step=1, description="random seed", unit=""),
    "celsius": dict(min=0, max=40, step=0.5, description="celsius", unit="\u00b0C"),
    # "v_init": dict(min=-90, max=-30, step=1, description="v_init", unit="mV"),
    "tstop": dict(min=10, max=1500, step=10, description="tstop", unit="ms"),
}

def build_epsc_dashboard():
    # only one preset for now, so no experiment selector
    defaults = DEFAULT_EPSC_PARAMS

    panel_checks, panels_row = _build_panel_checks(PLOT_PANELS_EPSC)
    sliders, changed_flags, rows = _build_param_rows(EPSC_PARAM_UI, lambda name: defaults[name])

    run_button = Button(description="Run", button_style="success", icon="play")
    reset_all_button = Button(description="Reset all", icon="undo")
    output = Output()

    def _on_run(_btn):
        with output:
            output.clear_output(wait=True)
            run_epsc_experiment(panels=_selected_panels(panel_checks), **_raw_values(EPSC_PARAM_UI, sliders))

    def _on_reset_all(_btn):
        for name, slider in sliders.items():
            slider.value = _display_value(EPSC_PARAM_UI, name, defaults[name])

    run_button.on_click(_on_run)
    reset_all_button.on_click(_on_reset_all)

    dashboard = VBox([HBox([VBox([panels_row, output]),
                             VBox([HBox([run_button, reset_all_button])] + rows)])])
    _on_run(None)  # show a default run immediately, without waiting for a click
    return dashboard

Adjust the stimulus/test spike trains and release parameters and click **Run**. Watch the
**vesicle pool** panel to see the readily-releasable pool deplete during the stimulus train and
recover during the test train — this is the mechanistic basis for the depression/facilitation
seen in the **EPSC** panel.

In [ ]:
display(build_epsc_dashboard())

# 2. EPSP mode (current clamp — postsynaptic spiking)

Same calyx-of-Held synapse, but `MNTB` now has active `BKDNaDR` (fast Na, delayed-rectifier K)
conductances instead of a voltage clamp, so postsynaptic potentials can summate and trigger
spikes. Sourced from `run_epsp.hoc` / `cohepsp.ses` (note the different resting potential,
-66 mV, and that the `COH (Globals)` panel isn't exposed here, matching the original session).

In [ ]:
DEFAULT_EPSP_PARAMS = {**DEFAULT_COH_PARAMS, **DEFAULT_SPIKE_PARAMS,
    "celsius": 36, "v_init": -66, "tstop": 100}

PLOT_PANELS_EPSP = {
    "v": dict(label="voltage", vector=v_vec, ylabel="membrane potential (mV)", ylim=(-90, 50)),
}

def run_epsp_experiment(panels=None, **overrides):
    p = {**DEFAULT_EPSP_PARAMS, **overrides}

    MNTB(0.5).g_pas = 1 / 500
    MNTB(0.5).e_pas = p["v_init"]
    MNTB(0.5).gnabar_BKDNaDR = 0.025
    MNTB(0.5).gkbar_BKDNaDR = 0.03

    vcl.dur[0] = 0  # inactive: EPSP mode must not be voltage-clamped

    _apply_coh_params(p)
    _set_spike_train(p)

    h.celsius = p["celsius"]
    h.tstop = p["tstop"]
    h.v_init = p["v_init"]
    h.run()

    _plot_panels(PLOT_PANELS_EPSP, panels)

In [ ]:
# @title Specifying default parameters

EPSP_PARAM_UI = {
    "stimdel": dict(min=0, max=50, step=1, description="stim delay", unit="ms"),
    "stimdur": dict(min=0, max=500, step=10, description="stim duration", unit="ms"),
    "stimfre": dict(min=1, max=300, step=1, description="stim frequency", unit="Hz"),
    "testdel": dict(min=0, max=50, step=1, description="test delay", unit="ms"),
    "testdur": dict(min=0, max=2000, step=10, description="test duration", unit="ms"),
    "testfre": dict(min=0, max=100, step=1, description="test frequency", unit="Hz"),
    "celsius": dict(min=0, max=40, step=0.5, description="celsius", unit="\u00b0C"),
    "v_init": dict(min=-90, max=-30, step=1, description="v_init", unit="mV"),
    "tstop": dict(min=10, max=1500, step=10, description="tstop", unit="ms"),
}

def build_epsp_dashboard():
    defaults = DEFAULT_EPSP_PARAMS

    panel_checks, panels_row = _build_panel_checks(PLOT_PANELS_EPSP)
    sliders, changed_flags, rows = _build_param_rows(EPSP_PARAM_UI, lambda name: defaults[name])

    run_button = Button(description="Run", button_style="success", icon="play")
    reset_all_button = Button(description="Reset all", icon="undo")
    output = Output()

    def _on_run(_btn):
        with output:
            output.clear_output(wait=True)
            run_epsp_experiment(panels=_selected_panels(panel_checks), **_raw_values(EPSP_PARAM_UI, sliders))

    def _on_reset_all(_btn):
        for name, slider in sliders.items():
            slider.value = _display_value(EPSP_PARAM_UI, name, defaults[name])

    run_button.on_click(_on_run)
    reset_all_button.on_click(_on_reset_all)

    dashboard = VBox([HBox([VBox([panels_row, output]),
                             VBox([HBox([run_button, reset_all_button])] + rows)])])
    _on_run(None)  # show a default run immediately, without waiting for a click
    return dashboard

Adjust the stimulus/test spike trains and click **Run** to see the postsynaptic voltage
response, including any spikes it triggers. The calyx's release-probability/replenishment
parameters (`COH (Globals)` in the original) aren't exposed here — as in the original
`cohepsp.ses` session, they stay at their built-in defaults; use the **EPSC mode** dashboard
above to explore those.

In [ ]:
display(build_epsp_dashboard())